# Chunking Strategies for RAG Pipelines

This notebook loads a real sample document from `data/sample_document.txt`
and runs it through five chunking strategies:

1. Fixed-size chunking
2. Sentence chunking
3. Paragraph chunking
4. Semantic chunking
5. Recursive chunking

Each strategy has its own code cell, followed by a markdown **verdict** cell.
The notebook ends with a comparison and a recommendation.

## Setup

Load the sample document and define shared helpers (token counting, sentence splitting).

In [1]:
import re

with open("data/sample_document.txt", "r", encoding="utf-8") as f:
    document = f.read()

print(f"Document length: {len(document)} characters, ~{len(document.split())} words")


Document length: 17257 characters, ~2676 words


In [2]:
try:
    import tiktoken
    _enc = tiktoken.get_encoding("cl100k_base")
    def count_tokens(text):
        return len(_enc.encode(text))
    def tokenize(text):
        return _enc.encode(text)
    def detokenize(tokens):
        return _enc.decode(tokens)
except ImportError:
    def count_tokens(text):
        return len(text.split())
    def tokenize(text):
        return text.split()
    def detokenize(tokens):
        return " ".join(tokens)

_SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+(?=[A-Z0-9])')

def split_sentences(text):
    return [s.strip() for s in _SENTENCE_SPLIT_RE.split(text.strip()) if s.strip()]

def preview_chunks(chunks, n=3, width=110):
    print(f"{len(chunks)} chunks total. Showing first {min(n, len(chunks))}:\n")
    for i, c in enumerate(chunks[:n], 1):
        flat = " ".join(c.split())
        print(f"[{i}] ({count_tokens(c)} tokens) {flat[:width]}{'...' if len(flat) > width else ''}\n")


## 1. Fixed-size chunking

Split by a fixed token count, with overlap so context cut at a boundary is still reachable from the neighboring chunk.

In [3]:
def fixed_size_chunking(text, chunk_size=512, overlap_pct=0.15):
    tokens = tokenize(text)
    overlap = int(chunk_size * overlap_pct)
    step = max(chunk_size - overlap, 1)

    chunks = []
    for start in range(0, len(tokens), step):
        window = tokens[start:start + chunk_size]
        if not window:
            break
        chunks.append(detokenize(window))
        if start + chunk_size >= len(tokens):
            break
    return chunks

fixed_chunks = fixed_size_chunking(document, chunk_size=200, overlap_pct=0.15)
preview_chunks(fixed_chunks)


16 chunks total. Showing first 3:

[1] (200 tokens) ISI’s role is to evaluate and understand the capabilities of frontier AI models, surfacing potential risks bef...

[2] (200 tokens) and organisations. In total, we catalogued 19 such actions. Almost all of this behaviour (17 actions) came fro...

[3] (200 tokens) conditions that do not reflect how frontier models are made available to the public. We do this to best assess...



## Fixed-size Chunking

This method produced 16 chunks, each capped at 200 tokens. The first three:

**Chunk 1** (200 tokens)
> ISI's role is to evaluate and understand the capabilities of frontier AI models, surfacing potential risks bef...

**Chunk 2** (200 tokens)
> and organisations. In total, we catalogued 19 such actions. Almost all of this behaviour (17 actions) came fro...

**Chunk 3** (200 tokens)
> conditions that do not reflect how frontier models are made available to the public. We do this to best assess...

**Observation:** Chunk sizes are perfectly uniform, but chunk 2 begins mid-sentence with "and organisations..." its meaning depends entirely on chunk 1, which isn't included alongside it. This illustrates the core limitation of fixed-size chunking: splits are based purely on token count, not on sentence or idea boundaries, so a cut can land anywhere.

## 2. Sentence chunking

Group N sentences per chunk. Never splits mid-sentence, but chunk size varies with sentence length.

In [4]:
def sentence_chunking(text, sentences_per_chunk=5, overlap_sentences=1):
    sentences = split_sentences(text)
    step = max(sentences_per_chunk - overlap_sentences, 1)

    chunks = []
    for start in range(0, len(sentences), step):
        group = sentences[start:start + sentences_per_chunk]
        if not group:
            break
        chunks.append(" ".join(group))
        if start + sentences_per_chunk >= len(sentences):
            break
    return chunks

sentence_chunks = sentence_chunking(document, sentences_per_chunk=4, overlap_sentences=1)
preview_chunks(sentence_chunks)


44 chunks total. Showing first 3:

[1] (99 tokens) ISI’s role is to evaluate and understand the capabilities of frontier AI models, surfacing potential risks bef...

[2] (72 tokens) On investigation, we found that some of the agents being tested had engaged in sustained, potentially harmful ...

[3] (70 tokens) We ran this challenge 122 times across several models. Our investigation found that in 10 of those runs, an AI...



This method produced 44 chunks, grouped by complete sentences. The first three:

**Chunk 1** (99 tokens)
> ISI's role is to evaluate and understand the capabilities of frontier AI models, surfacing potential risks bef...

**Chunk 2** (72 tokens)
> On investigation, we found that some of the agents being tested had engaged in sustained, potentially harmful ...

**Chunk 3** (70 tokens)
> We ran this challenge 122 times across several models. Our investigation found that in 10 of those runs, an AI...

**Observation:** Every chunk begins and ends on a complete sentence, unlike fixed-size chunking. However, token counts vary noticeably between chunks (99, 72, 70), since sentence length in this document is inconsistent. This reflects the core trade-off of sentence chunking: sentence boundaries are respected, but chunk size is not controlled.

## 3. Paragraph chunking

Split on blank lines. Falls back to fixed-size chunking for any single paragraph that's still too large.

In [5]:
def paragraph_chunking(text, max_tokens=250):
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

    chunks = []
    for p in paragraphs:
        if count_tokens(p) <= max_tokens:
            chunks.append(p)
        else:
            chunks.extend(fixed_size_chunking(p, chunk_size=max_tokens, overlap_pct=0.1))
    return chunks

paragraph_chunks = paragraph_chunking(document, max_tokens=250)
preview_chunks(paragraph_chunks)


46 chunks total. Showing first 3:

[1] (55 tokens) ISI’s role is to evaluate and understand the capabilities of frontier AI models, surfacing potential risks bef...

[2] (64 tokens) On 28th July 2026, AISI's Security Team detected unusual data transfers leaving our research systems during a ...

[3] (147 tokens) The incident stemmed from a single evaluation where agents were given a task of solving a cyber security chall...



## Paragraph Chunking

This method produced 46 chunks, split along the document's natural paragraph breaks. The first three:

**Chunk 1** (55 tokens)
> ISI's role is to evaluate and understand the capabilities of frontier AI models, surfacing potential risks bef...

**Chunk 2** (64 tokens)
> On 28th July 2026, AISI's Security Team detected unusual data transfers leaving our research systems during a ...

**Chunk 3** (147 tokens)
> The incident stemmed from a single evaluation where agents were given a task of solving a cyber security chall...

**Observation:** Chunks follow the author's own paragraph structure, so each one tends to carry a single, self-contained point. Token counts still vary (55, 64, 147), since paragraph length in the source document is inconsistent, some paragraphs are short transitional statements, others run long. This method preserves natural boundaries well, but does not guarantee a consistent chunk size.

## 4. Semantic Chunking

Sentences are embedded using *`sentence-transformers`* (*`all-MiniLM-L6-v2`*), then merged into a chunk as long as the similarity between consecutive sentences stays above a threshold; a similarity drop starts a new chunk.


In [6]:
import math
from sentence_transformers import SentenceTransformer

_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_sentences(sentences):
    return _model.encode(sentences)

def cosine_sim(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

def semantic_chunking(text, similarity_threshold=0.15, max_tokens=250):
    sentences = split_sentences(text)
    if len(sentences) <= 1:
        return sentences

    embeddings = embed_sentences(sentences)

    chunks = []
    current = [sentences[0]]
    current_tokens = count_tokens(sentences[0])

    for i in range(1, len(sentences)):
        sim = cosine_sim(embeddings[i - 1], embeddings[i])
        next_tokens = count_tokens(sentences[i])

        if sim >= similarity_threshold and current_tokens + next_tokens <= max_tokens:
            current.append(sentences[i])
            current_tokens += next_tokens
        else:
            chunks.append(" ".join(current))
            current = [sentences[i]]
            current_tokens = next_tokens

    if current:
        chunks.append(" ".join(current))
    return chunks

semantic_chunks = semantic_chunking(document, similarity_threshold=0.1, max_tokens=250)
preview_chunks(semantic_chunks)

/home/as/Documents/GitHub/chunking_strategy/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1310.24it/s]


21 chunks total. Showing first 3:

[1] (224 tokens) ISI’s role is to evaluate and understand the capabilities of frontier AI models, surfacing potential risks bef...

[2] (244 tokens) In an attempt to get the code approved, the agent engaged in social engineering — creating fake online identit...

[3] (40 tokens) We also intend to work with METR (Model Evaluation and Threat Research) to conduct an independent third-party ...




**This method produced 21 chunks.**

The first three:

**Chunk 1** (224 tokens)

> ISI's role is to evaluate and understand the capabilities of frontier AI models, surfacing potential risks bef...

**Chunk 2** (244 tokens)

> In an attempt to get the code approved, the agent engaged in social engineering — creating fake online identit...

**Chunk 3** (40 tokens)

> We also intend to work with METR (Model Evaluation and Threat Research) to conduct an independent third-party ...

**Observation:** Chunk 3 is notably shorter (40 tokens) than chunks 1 and 2 (224, 244 tokens). This is expected because a new chunk starts when the similarity between consecutive sentences falls below the threshold. A similarity threshold of 0.1 used here. Therefore, chunk length depends on topic continuity rather than a fixed size.

## 5. Recursive chunking

In [7]:
def recursive_chunking(text, chunk_size=250, overlap_pct=0.15, separators=None):
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    def _split(text, seps):
        if count_tokens(text) <= chunk_size:
            return [text] if text.strip() else []

        if not seps:
            return fixed_size_chunking(text, chunk_size, 0)

        sep, rest = seps[0], seps[1:]
        pieces = text.split(sep) if sep else list(text)

        result = []
        current = ""

        for piece in pieces:
            candidate = current + sep + piece if current else piece

            if count_tokens(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    result.append(current)

                if count_tokens(piece) > chunk_size:
                    result.extend(_split(piece, rest))
                    current = ""
                else:
                    current = piece

        if current:
            result.append(current)

        return result

    chunks = _split(text.strip(), separators)

    # Apply overlap 
    if overlap_pct > 0 and len(chunks) > 1:
        overlapped = [chunks[0]]

        for i in range(1, len(chunks)):
            current_tokens = tokenize(chunks[i])

            overlap_tokens = int(chunk_size * overlap_pct)

            # Reserve space for overlap
            max_new_tokens = chunk_size - overlap_tokens

            if i == len(chunks) - 1:
                new_tokens = current_tokens
            else:
                new_tokens = current_tokens[:max_new_tokens]

            # Take overlap from previous chunk
            prev_tokens = tokenize(chunks[i - 1])
            overlap_text = detokenize(prev_tokens[-overlap_tokens:])

            combined = (overlap_text + " " + detokenize(new_tokens)).strip()

            overlapped.append(combined)

        chunks = overlapped

    return chunks

In [8]:
recursive_chunks = recursive_chunking(
    document,
    chunk_size=250,
    overlap_pct=0.15
)

preview_chunks(recursive_chunks)

14 chunks total. Showing first 3:

[1] (119 tokens) ISI’s role is to evaluate and understand the capabilities of frontier AI models, surfacing potential risks bef...

[2] (221 tokens) the agents being tested had engaged in sustained, potentially harmful activity directed at real people and org...

[3] (250 tokens) These attempts were unsuccessful, and our investigations have not evidenced any resulting real-world harm. But...



In [9]:
sizes = [count_tokens(c) for c in recursive_chunks]

print("Number of chunks:", len(recursive_chunks))
print("Maximum tokens:", max(sizes))
print("Minimum tokens:", min(sizes))
print("Average tokens:", sum(sizes) / len(sizes))

Number of chunks: 14
Maximum tokens: 250
Minimum tokens: 105
Average tokens: 220.0


Splits are attempted using separators in order of structural strength, paragraph, then sentence, then word, then character — recursively splitting only the pieces still too large. This method produced 14 chunks. The first three:

**Chunk 1** (119 tokens)

> ISI's role is to evaluate and understand the capabilities of frontier AI models, surfacing potential risks bef...

**Chunk 2** (201 tokens)

> security incident and, within roughly one hour of discovery, had contained it and begun a full investigation. ...

**Chunk 3** (250 tokens)

> any resulting real-world harm. But this is the first time we have seen risks around autonomy and deception man...

**Observation:** This method produced the fewest chunks (14) of all five approaches, with sizes trending closer to the target budget than sentence or paragraph chunking managed. It keeps chunk size close to the target, with a maximum of 250 tokens. Compared to fixed-size chunking, it avoids arbitrary mid-sentence cuts in most cases while still keeping chunk size close to a set target, without requiring an embedding model.


## Comparison

In [10]:
import statistics as stats

results = {
    "Fixed-size": fixed_chunks,
    "Sentence": sentence_chunks,
    "Paragraph": paragraph_chunks,
    "Semantic": semantic_chunks,
    "Recursive": recursive_chunks,
}

print(f"{'Strategy':<12} {'# chunks':>9} {'avg tokens':>11} {'std dev':>9} {'min':>6} {'max':>6}")
for name, chunks in results.items():
    sizes = [count_tokens(c) for c in chunks]
    print(f"{name:<12} {len(chunks):>9} {stats.mean(sizes):>11.1f} "
          f"{(stats.pstdev(sizes) if len(sizes) > 1 else 0):>9.1f} {min(sizes):>6} {max(sizes):>6}")


Strategy      # chunks  avg tokens   std dev    min    max
Fixed-size          16       195.4      17.9    126    200
Sentence            44        80.3      22.8     31    123
Paragraph           46        58.2      27.7     17    147
Semantic            21       127.4      94.3      1    248
Recursive           14       220.0      45.6    105    250


## Final Verdicts and Recommendation

| Strategy   | # Chunks | Avg Tokens | Std Dev | Min | Max |
| ---------- | -------: | ---------: | ------: | --: | --: |
| Fixed-size |       16 |      195.4 |    17.9 | 126 | 200 |
| Sentence   |       44 |       80.3 |    22.8 |  31 | 123 |
| Paragraph  |       46 |       58.2 |    27.7 |  17 | 147 |
| Semantic   |       21 |      127.4 |    94.3 |   1 | 248 |
| Recursive  |       14 |      220.0 |    45.6 | 105 | 250 |

### Verdicts

* **Fixed-size:** Most consistent chunk sizes, but it can cut across sentences or ideas. Good for size control, but less aware of meaning.
* **Sentence:** Preserves complete sentences, but produces many small chunks and increases the number of chunks for retrieval.
* **Paragraph:** Preserves the document structure, but produces many small chunks and has considerable variation in size.
* **Semantic:** Groups content based on topic similarity, but has high variation in chunk size, including very small chunks. It also requires an embedding model.
* **Recursive:** Produces the fewest chunks and keeps chunks close to the target size while preserving paragraph and sentence boundaries where possible.

### Recommendation

**Recursive chunking is the best fit for this use case.** It provides a good balance between chunk size, document structure, and number of chunks without requiring an embedding model during chunking.

Semantic chunking can be considered as an alternative for comparison, but its high variation in chunk size makes it less suitable as the default approach for this document.
